# Demo 5 Setup

This notebook creates the schema and tables used in **Demo 5: DAX vs SQL Examples**.

In [0]:
# Get current user and set up schema name
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]
schema_name = f"demo_dax_sql_{clean_username}"

# Create the schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}` COMMENT 'Demo schema for DAX vs SQL examples'")
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{schema_name}`")

print(f"\u2713 Schema ready: {catalog_name}.{schema_name}")

In [0]:
# Generate gold_monthly_sales data
# 24 months (Jan 2023 - Dec 2024), 4 regions, 5 product categories
import random
from pyspark.sql import Row
from datetime import date

random.seed(42)

regions = ['North', 'South', 'East', 'West']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Food & Beverage', 'Sports']

# Base revenue by region and category (creates realistic variation)
base_revenue = {
    ('North', 'Electronics'): 520000, ('North', 'Clothing'): 310000,
    ('North', 'Home & Garden'): 280000, ('North', 'Food & Beverage'): 420000,
    ('North', 'Sports'): 195000,
    ('South', 'Electronics'): 480000, ('South', 'Clothing'): 350000,
    ('South', 'Home & Garden'): 260000, ('South', 'Food & Beverage'): 390000,
    ('South', 'Sports'): 220000,
    ('East', 'Electronics'): 610000, ('East', 'Clothing'): 290000,
    ('East', 'Home & Garden'): 310000, ('East', 'Food & Beverage'): 370000,
    ('East', 'Sports'): 180000,
    ('West', 'Electronics'): 570000, ('West', 'Clothing'): 340000,
    ('West', 'Home & Garden'): 300000, ('West', 'Food & Beverage'): 410000,
    ('West', 'Sports'): 240000,
}

# Seasonality multiplier by month (1=Jan, 12=Dec)
seasonality = {
    1: 0.85, 2: 0.80, 3: 0.90, 4: 0.95, 5: 1.00, 6: 1.05,
    7: 1.02, 8: 0.98, 9: 1.00, 10: 1.05, 11: 1.15, 12: 1.30
}

rows = []
for year in [2023, 2024]:
    # Year-over-year growth factor (2024 is ~8% growth over 2023)
    yoy_factor = 1.0 if year == 2023 else 1.08
    for month in range(1, 13):
        for region in regions:
            for category in categories:
                base = base_revenue[(region, category)]
                # Apply seasonality, YoY growth, and random variation
                revenue = base * seasonality[month] * yoy_factor * random.uniform(0.88, 1.12)
                units = int(revenue / random.uniform(45, 85))
                avg_val = round(revenue / units, 2)
                rows.append(Row(
                    report_month=date(year, month, 1),
                    region=region,
                    product_category=category,
                    total_revenue=round(revenue, 2),
                    total_units_sold=units,
                    avg_order_value=avg_val
                ))

df_sales = spark.createDataFrame(rows)
print(f"\u2713 Generated {df_sales.count()} rows of monthly sales data")

In [0]:
# Add yoy_growth_pct by comparing each month to the same month in the prior year
from pyspark.sql import Window
from pyspark.sql import functions as F

# Self-join to get prior year revenue
df_current = df_sales.alias("curr")
df_prior = df_sales.alias("prior")

df_with_yoy = df_current.join(
    df_prior,
    (F.col("curr.region") == F.col("prior.region")) &
    (F.col("curr.product_category") == F.col("prior.product_category")) &
    (F.col("curr.report_month") == F.add_months(F.col("prior.report_month"), 12)),
    "left"
).select(
    F.col("curr.report_month"),
    F.col("curr.region"),
    F.col("curr.product_category"),
    F.col("curr.total_revenue"),
    F.col("curr.total_units_sold"),
    F.col("curr.avg_order_value"),
    F.round(
        (F.col("curr.total_revenue") - F.col("prior.total_revenue")) / F.col("prior.total_revenue") * 100, 1
    ).alias("yoy_growth_pct")
)

# Write to table
df_with_yoy.write.mode("overwrite").saveAsTable("gold_monthly_sales")

count = spark.sql("SELECT COUNT(*) FROM gold_monthly_sales").first()[0]
print(f"\u2713 Table created: gold_monthly_sales ({count:,} rows)")

In [0]:
# Create dim_customers table
customer_rows = []
segments = ['Enterprise', 'Mid-Market', 'Small Business']

random.seed(99)
first_names = ['James', 'Sarah', 'Michael', 'Emma', 'David', 'Lisa', 'Robert', 'Jennifer',
               'William', 'Maria', 'Thomas', 'Patricia', 'Daniel', 'Linda', 'Andrew']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller',
              'Davis', 'Rodriguez', 'Martinez', 'Wilson', 'Anderson', 'Taylor', 'Thomas']

cust_id = 1
for region in regions:
    for segment in segments:
        # Different number of customers per segment
        n_customers = {'Enterprise': 3, 'Mid-Market': 5, 'Small Business': 8}[segment]
        for _ in range(n_customers):
            name = f"{random.choice(first_names)} {random.choice(last_names)}"
            customer_rows.append(Row(
                customer_id=cust_id,
                customer_name=name,
                region=region,
                segment=segment
            ))
            cust_id += 1

df_customers = spark.createDataFrame(customer_rows)
df_customers.write.mode("overwrite").saveAsTable("dim_customers")

count = spark.sql("SELECT COUNT(*) FROM dim_customers").first()[0]
print(f"\u2713 Table created: dim_customers ({count:,} rows)")

In [0]:
# Add table and column comments
spark.sql("COMMENT ON TABLE gold_monthly_sales IS 'Monthly aggregated sales data by region and product category (Jan 2023 - Dec 2024)'")
spark.sql("COMMENT ON TABLE dim_customers IS 'Customer dimension table with segment classification'")

spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN report_month COMMENT 'First day of the reporting month'")
spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN region COMMENT 'Sales region (North, South, East, West)'")
spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN product_category COMMENT 'Product category name'")
spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN total_revenue COMMENT 'Total revenue for the month/region/category'")
spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN total_units_sold COMMENT 'Total units sold'")
spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN avg_order_value COMMENT 'Average order value'")
spark.sql("ALTER TABLE gold_monthly_sales ALTER COLUMN yoy_growth_pct COMMENT 'Year-over-year growth percentage (NULL for 2023)'")

print("\u2713 Comments added")

In [0]:
# Verify the data looks good
print("\n" + "="*60)
print("SAMPLE DATA: gold_monthly_sales")
print("="*60)
display(spark.sql("SELECT * FROM gold_monthly_sales ORDER BY report_month DESC, total_revenue DESC LIMIT 10"))

In [0]:
print("")
print("="*60)
print("\u2713 SETUP COMPLETE")
print("="*60)
print(f"")
print(f"Catalog:  {catalog_name}")
print(f"Schema:   {schema_name}")
print(f"")
print(f"Tables created:")
print(f"  \u2022 gold_monthly_sales  - 480 rows (24 months \u00d7 4 regions \u00d7 5 categories)")
print(f"  \u2022 dim_customers       - 64 customers across 4 regions and 3 segments")
print(f"")
print(f"The SQL cells in this demo use the current schema context.")
print(f"All queries reference tables without a catalog/schema prefix.")